In [ ]:
#uv add optuna

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import optuna
from torchvision import datasets,models,transforms
import numpy as np

In [2]:
transform_train = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

    ]
)

transform_test = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]
)

In [3]:
train_datasets = datasets.ImageFolder(root='./dataset/train',transform=transform_train)
test_datasets = datasets.ImageFolder(root='./dataset/test',transform=transform_test)

In [4]:
model = models.resnet34(pretrained=True)
model.fc = nn.Linear(512,3)

c:\potenup3\prj_deep\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\potenup3\prj_deep\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
from torch.utils.data import random_split
train_data_size = len(train_datasets)

#8:2
train_size = int(train_data_size*0.8)
val_size = train_data_size - train_size

train_dataset,val_dataset = random_split(train_datasets,[train_size,val_size])


In [10]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = models.resnet34(pretrained=True)
model.fc = nn.Linear(512,3)
model = model.to(device)

In [16]:
def objective(trial):
    batch_size = trial.suggest_categorical('batch_size',[4,6,8])
    lr = trial.suggest_loguniform('lr',1e-5,1e-3)
    train_dataloader = torch.utils.data.DataLoader(train_datasets, shuffle=True, batch_size=batch_size)
    val_dadtaloader = torch.utils.data.DataLoader(test_datasets, shuffle=True, batch_size=batch_size)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    optimizer = optim.Adam(model.parameters(), lr = lr)
    criterion = nn.CrossEntropyLoss()
    epochs = 20
    val_loss = 0

    for epoch in range(epochs):
        model.train()

        for img,labels in train_dataloader:
            optimizer.zero_grad()
            preds = model(img.to(device))
            loss = criterion(preds,labels.to(device))
            loss.backward()
            optimizer.step()

        # 중간검증
        model.eval()
        with torch.no_grad():
            for img,labels in val_dadtaloader:
                #preds = model(img.to(device))
                labels = labels.to(device)
                img = img.to(device)
                preds = model(img)
                val_loss += criterion(preds,labels)

        total_loss = val_loss / len(val_dadtaloader)

        trial.report(total_loss,epoch)

        if trial.should_prune(): 
            raise optuna.exceptions.TrialPruned()
        
    return total_loss

study = optuna.create_study(direction='minimize')
study.optimize(objective,n_trials=50) # 50개의 경우의 수로 학습

[I 2026-03-06 10:54:39,463] A new study created in memory with name: no-name-ad30c0ae-6528-489c-b4e2-ab2b1fd8f69a
C:\Users\user\AppData\Local\Temp\ipykernel_13608\1409558086.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr',1e-5,1e-3)
[I 2026-03-06 10:56:03,783] Trial 0 finished with value: 10.14249324798584 and parameters: {'batch_size': 6, 'lr': 6.0427978842080025e-05}. Best is trial 0 with value: 10.14249324798584.
[I 2026-03-06 10:57:32,800] Trial 1 finished with value: 22.800806045532227 and parameters: {'batch_size': 4, 'lr': 0.0006294152363194208}. Best is trial 0 with value: 10.14249324798584.
[I 2026-03-06 10:59:01,841] Trial 2 finished with value: 8.46710205078125 and parameters: {'batch_size': 4, 'lr': 1.7548610450748045e-05}. Best is trial 2 with value: 8.46710205078125.
[I 2026-

In [15]:
print(study.best_trial.params)

{'batch_size': 8, 'lr': 1.756479236609023e-05}


명소 : 거리,테마공원
문화 : 공연장,문화예술공간
문화유산 : 문화재,보물,사적,성
시설 : 시장
자연 : 국립공원,